<a href="https://colab.research.google.com/github/Akhila-010/GenAI_Tasks/blob/main/College_RAG_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import getpass
from openai import OpenAI

OPENAI_API_KEY = getpass.getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=OPENAI_API_KEY)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello"}]
)

print(response.choices[0].message.content)

Enter your OpenAI API key: ··········
Hello! How can I assist you today?


In [ ]:
!pip install openai faiss-cpu PyPDF2 gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 12.8 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving information-brochure.pdf to information-brochure.pdf


In [ ]:
import PyPDF2

def extract_text(file):
    reader = PyPDF2.PdfReader(file)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

pdf_file = list(uploaded.keys())[0]
text = extract_text(pdf_file)

In [ ]:
def chunk_text(text, size=500, overlap=100):
    chunks = []
    for i in range(0, len(text), size - overlap):
        chunks.append(text[i:i+size])
    return chunks

chunks = chunk_text(text)
print("Total chunks:", len(chunks))

Total chunks: 358


In [ ]:
def get_embeddings(chunks):
    embeddings = []
    for chunk in chunks:
        res = client.embeddings.create(
            model="text-embedding-3-small",
            input=chunk
        )
        embeddings.append(res.data[0].embedding)
    return embeddings

embeddings = get_embeddings(chunks)

In [ ]:
import faiss
import numpy as np

embeddings_np = np.array(embeddings).astype("float32")

index = faiss.IndexFlatL2(len(embeddings_np[0]))
index.add(embeddings_np)

In [ ]:
def search(query, k=3):
    q_embed = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    ).data[0].embedding

    D, I = index.search(np.array([q_embed]).astype("float32"), k)
    return [chunks[i] for i in I[0]]

In [ ]:
def ask(query):
    context = "\n".join(search(query))

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer from the context only."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion:{query}"}
        ]
    )

    return response.choices[0].message.content

In [ ]:
chat_history = []

while True:
    query = input("\nYou: ")

    if query.lower() == "exit":
        print("Chat ended.")
        break

    chat_history.append({"role": "user", "content": query})

    context = "\n".join(search(query))

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer from the context only."},
            *chat_history,
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion:{query}"}
        ]
    )

    answer = response.choices[0].message.content
    chat_history.append({"role": "assistant", "content": answer})

    print("\nBot:", answer)


You: what is the fee structure ?

Bot: The fee structure is as follows for M.Tech./M.S.(R)/M.Des. students receiving Institute/Project Assistantship or Teaching positions:

- Tuition Fees: Rs. 10,000
- Other Charges: Rs. 164
- Hostel Seat Rent & Amenity Charges: Specific amounts not provided for hostel accommodation.
- Total Amount: Not explicitly stated, but calculated based on the values above.

For all other candidates, the application fee is Rs. 1,500.

You: what are the branches avaliable ?

Bot: The available branches include:

1. Atmospheric and Oceanic Sciences
2. Meteorology
3. Remote Sensing
4. M.Sc. Meteorology
5. M.Sc. Atmospheric Sciences
6. M.Sc. Oceanography
7. M.Sc. Physics
8. M.Sc. Geo-Physics
9. M.Sc. Marine Engineering
10. M.Sc. Statistics
11. M.Sc. Chemistry
12. M.A./M.Sc. Maths
13. MCA
14. M.Sc. Environmental Sciences
15. M.Sc. Agricultural Sciences
16. M.Sc. Geo-Information/Remote Sensing

Engineering branches available include:

1. Aeronautical Engineering
2. Au

In [ ]:
import gradio as gr

gr.Interface(fn=ask, inputs="text", outputs="text",
             title="College Assistant").launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5c67e1b1219c3de44c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
